In [44]:
import warnings
warnings.filterwarnings("ignore")        # suppress noisy convergence warnings

import os
import joblib                            # to save/load trained model objects
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Sklearn: splitting & evaluation ---
from sklearn.model_selection import (
    StratifiedKFold,          # 5-fold CV that keeps class ratios balanced
    cross_validate,           # runs CV and returns multiple score metrics
    GridSearchCV,             # exhaustive hyperparameter search
)

# --- Sklearn: preprocessing ---
from sklearn.preprocessing import (
    LabelEncoder,             # converts class names  →  integers
    StandardScaler,           # rescales features to mean=0, std=1
    OrdinalEncoder,           # encodes text categories  →  numeric codes
)
from sklearn.impute import SimpleImputer  # fills missing values

# --- Sklearn: pipeline & column routing ---
from sklearn.pipeline  import Pipeline          # chains steps sequentially
from sklearn.compose   import ColumnTransformer # routes columns to different steps

# --- Sklearn: feature selection ---
from sklearn.feature_selection import (
    VarianceThreshold,        # drops near-constant / zero-variance features
    SelectKBest,              # keeps top-k features by a scoring function
    f_classif,                # ANOVA F-score: measures class separability per feature
)

# --- Sklearn: classifiers ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble     import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,  # XGBoost alternative — built into sklearn,
                                     # zero extra install needed, handles missing
                                     # values natively, same boosting power
)
from sklearn.svm import SVC

# --- Sklearn: metrics (for training-set report only) ---
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)

In [45]:
df_train = pd.read_csv("/Volumes/Samsung_T5/first_iterations/train_66_ros.csv")

print(f"  Rows    : {df_train.shape[0]}")
print(f"  Columns : {df_train.shape[1]}")
print(f"\n  Class distribution (disease_y):")
print(df_train["disease_y"].value_counts().to_string())
print()


  Rows    : 10940
  Columns : 73

  Class distribution (disease_y):
disease_y
healthy    4040
asthma     2500
covid      2200
copd       2200



In [46]:

TARGET = "disease_y"

X_train_raw = df_train.drop(columns=[TARGET]).copy()
y_train_raw = df_train[TARGET].copy()

print(f"  Feature matrix shape  : {X_train_raw.shape}")
print(f"  Target series shape   : {y_train_raw.shape}")
print()


  Feature matrix shape  : (10940, 72)
  Target series shape   : (10940,)



In [47]:
print("=" * 65)
print("STEP 4 : Identifying column types")
print("=" * 65)

bool_cols = X_train_raw.select_dtypes(include="bool").columns.tolist()
cat_cols  = X_train_raw.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols  = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()

print(f"  Boolean columns     : {bool_cols}")
print(f"  Categorical columns : {cat_cols}")
print(f"  Numeric columns     : {len(num_cols)} columns")
print(f"\n  Missing value counts:")
missing = X_train_raw.isnull().sum()
print(missing[missing > 0].to_string())
print()


STEP 4 : Identifying column types
  Boolean columns     : ['cough_present', 'fever_present']
  Categorical columns : ['gender', 'smoker', 'cold_present']
  Numeric columns     : 67 columns

  Missing value counts:
smoker          3530
cold_present    1605



In [48]:
# =============================================================================
# STEP 5 — FIX BOOLEAN COLUMNS
# =============================================================================
# sklearn models internally treat all data as floats.
# Python booleans (True/False) need to be converted to 1.0 / 0.0 explicitly,
# otherwise some transformers may throw dtype errors.

print("=" * 65)
print("STEP 5 : Converting boolean columns to float (True→1.0, False→0.0)")
print("=" * 65)

X_train_raw[bool_cols] = X_train_raw[bool_cols].astype(float)

# After conversion, add them into num_cols so they get scaled alongside numbers
for col in bool_cols:
    if col not in num_cols:
        num_cols.append(col)

print(f"  Converted : {bool_cols}")
print(f"  Updated numeric column count : {len(num_cols)}")
print()


STEP 5 : Converting boolean columns to float (True→1.0, False→0.0)
  Converted : ['cough_present', 'fever_present']
  Updated numeric column count : 69



In [49]:
# =============================================================================
# STEP 6 — ENCODE THE TARGET LABELS
# =============================================================================
# LabelEncoder converts string class names to integers:
#   asthma → 0,  copd → 1,  covid → 2,  healthy → 3   (alphabetical)
# Several sklearn models work better with integer targets. The mapping is
# saved so we can decode predictions back to human-readable names later.

print("=" * 65)
print("STEP 6 : Encoding target labels to integers")
print("=" * 65)

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)   # returns a numpy array of integers

print("  Class → Integer mapping:")
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f"    {cls:10s} → {idx}")
print()

STEP 6 : Encoding target labels to integers
  Class → Integer mapping:
    asthma     → 0
    copd       → 1
    covid      → 2
    healthy    → 3



In [50]:
# STEP 7 — BUILD THE PREPROCESSING PIPELINE
# =============================================================================
# We build two sub-pipelines:
#
#   num_transformer  → for all numeric + boolean columns
#     1. SimpleImputer(median)  : fills NaN with the column median value
#                                 (median is robust to outliers unlike mean)
#     2. StandardScaler         : rescales each feature to mean=0, std=1
#                                 (essential for SVM and Logistic Regression)
#
#   cat_transformer  → for categorical text columns (just "gender" here)
#     1. SimpleImputer(most_frequent) : fills NaN with the most common value
#     2. OrdinalEncoder               : encodes "female"/"male" → 0.0 / 1.0
#
#   ColumnTransformer forks the data, applies each transformer to its
#   designated columns, and reassembles everything into one matrix.

print("=" * 65)
print("STEP 7 : Building the preprocessing pipeline")
print("=" * 65)

# --- numeric sub-pipeline ---
num_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median"),
        # WHY median: 'smoker' and 'cold_present' are binary (0/1) columns
        # with ~32% and ~15% missing. Median preserves the 0/1 nature better
        # than mean (which would give a decimal like 0.22).
    ),
    (
        "scaler",
        StandardScaler(),
        # WHY scale: MFCC features span wildly different ranges
        # (mfcc_1_mean ~ -192, zcr_mean ~ 0.001). Without scaling, features
        # with large values dominate distance-based models like SVM.
    ),
])

# --- categorical sub-pipeline ---
cat_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent"),
    ),
    (
        "encoder",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            # handle_unknown: if a new unseen category appears at inference
            # time, encode it as -1 instead of crashing.
        ),
    ),
])

# --- combine both into one ColumnTransformer ---
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("cat", cat_transformer, cat_cols),
    ],
    remainder="drop",   # drop any leftover columns (there shouldn't be any)
)

print("  num_transformer  → impute (median) + StandardScaler")
print(f"    applies to {len(num_cols)} numeric/bool columns")
print("  cat_transformer  → impute (most_frequent) + OrdinalEncoder")
print(f"    applies to {len(cat_cols)} categorical columns : {cat_cols}")
print()

STEP 7 : Building the preprocessing pipeline
  num_transformer  → impute (median) + StandardScaler
    applies to 69 numeric/bool columns
  cat_transformer  → impute (most_frequent) + OrdinalEncoder
    applies to 3 categorical columns : ['gender', 'smoker', 'cold_present']



In [51]:
# =============================================================================
# STEP 8 — BUILD THE FEATURE SELECTION PIPELINE
# =============================================================================
# Even after preprocessing, we have 72 features — many of which are correlated
# MFCC variants.  Keeping all of them risks:
#   • overfitting (model memorises noise instead of signal)
#   • slower training
#   • reduced interpretability
#
# We apply two-stage trimming:
#
#   Stage 1 — VarianceThreshold(threshold=0.01)
#     Drops any feature whose variance is less than 0.01 after scaling.
#     After StandardScaler, variance=1 for all features normally, but if a
#     feature was near-constant BEFORE scaling, scaling amplifies noise —
#     so we catch them at this threshold.
#
#   Stage 2 — SelectKBest(f_classif, k=40)
#     Scores every remaining feature with the ANOVA F-test:
#     "How much does this feature's value differ across the 4 disease classes?"
#     Higher F-score = more discriminative = more useful for classification.
#     We keep the top 40 (out of ~72), discarding the weakest 32.
#     k=40 is a starting point — GridSearchCV will also tune this value.

print("=" * 65)
print("STEP 8 : Building the feature selection pipeline")
print("=" * 65)

feature_selection = Pipeline(steps=[
    (
        "var_filter",
        VarianceThreshold(threshold=0.01),
        # Drops near-constant features.  After StandardScaler these are
        # features that were essentially identical across all samples.
    ),
    (
        "kbest",
        SelectKBest(score_func=f_classif, k=40),
        # ANOVA F-test: tests whether a feature's mean differs significantly
        # across disease classes.  Keeps the 40 most class-separating features.
    ),
])

print("  Stage 1 : VarianceThreshold(0.01)  → drops near-constant features")
print("  Stage 2 : SelectKBest(f_classif, k=40) → keeps top 40 by ANOVA F-score")
print()



STEP 8 : Building the feature selection pipeline
  Stage 1 : VarianceThreshold(0.01)  → drops near-constant features
  Stage 2 : SelectKBest(f_classif, k=40) → keeps top 40 by ANOVA F-score



In [52]:
# =============================================================================
# STEP 9 — HELPER: BUILD FULL END-TO-END PIPELINE PER MODEL
# =============================================================================
# Each model gets wrapped in the SAME pipeline structure:
#   preprocessor  →  feature_selection  →  classifier
#
# Using a Pipeline here is critical: during cross-validation, the scaler and
# feature selector are fit ONLY on the training folds and applied to the
# validation fold — preventing data leakage (the #1 mistake in ML).

def build_full_pipeline(clf):
    """
    Wrap a classifier in the full preprocessing + feature selection pipeline.

    Parameters
    ----------
    clf : sklearn-compatible classifier object

    Returns
    -------
    sklearn Pipeline ready to call .fit(X, y)
    """
    return Pipeline(steps=[
        ("prep",    preprocessor),       # impute + scale + encode
        ("featsel", feature_selection),  # variance filter + SelectKBest
        ("clf",     clf),                # the actual classifier
    ])

print("=" * 65)
print("STEP 9 : Helper function build_full_pipeline() defined")
print("=" * 65)
print("  Each model will be wrapped as: preprocessor → featsel → classifier")
print()

STEP 9 : Helper function build_full_pipeline() defined
  Each model will be wrapped as: preprocessor → featsel → classifier



In [53]:
print("=" * 65)
print("STEP 10 : Defining candidate models")
print("=" * 65)

models = {}

# --- Model 1: Logistic Regression ---
models["LogisticRegression"] = build_full_pipeline(
    LogisticRegression(
        C=1.0,                      # inverse regularisation strength; C=1 = mild L2 penalty
        max_iter=2000,              # enough iterations to converge on 40 features
        multi_class="multinomial",  # native multi-class (4 diseases) via softmax
        class_weight="balanced",    # auto-upweights minority classes
        solver="lbfgs",             # efficient solver for multinomial + L2
        random_state=42,
    )
)
print("  [1] LogisticRegression — linear baseline with L2 regularisation")

# --- Model 2: Random Forest ---
models["RandomForest"] = build_full_pipeline(
    RandomForestClassifier(
        n_estimators=200,       # 200 trees — good accuracy/speed tradeoff
        max_depth=None,         # trees grow fully; forest's bagging prevents overfit
        min_samples_leaf=2,     # each leaf needs ≥2 samples — slight smoothing
        class_weight="balanced",
        n_jobs=-1,              # use all CPU cores
        random_state=42,
    )
)
print("  [2] RandomForest — 200 trees, bagging, feature importance available")

# --- Model 3: SVM with RBF kernel ---
models["SVM_RBF"] = build_full_pipeline(
    SVC(
        kernel="rbf",           # radial basis function — non-linear boundary
        C=10,                   # soft margin; allows some misclassifications
        gamma="scale",          # gamma = 1/(n_features * X.var()) — auto-scaled
        probability=True,       # enables .predict_proba() at inference time
        class_weight="balanced",
        random_state=42,
    )
)
print("  [3] SVM (RBF) — non-linear kernel, good for high-dim audio features")

# --- Model 4: Gradient Boosting ---
models["GradientBoosting"] = build_full_pipeline(
    GradientBoostingClassifier(
        n_estimators=200,       # 200 sequential trees
        learning_rate=0.1,      # step size per tree (shrinkage)
        max_depth=5,            # max depth per tree — controls complexity
        subsample=0.8,          # use 80% of samples per tree — reduces variance
        random_state=42,
    )
)
print("  [4] GradientBoosting — sequential tree boosting, sklearn native")

# --- Model 5: HistGradientBoosting (XGBoost alternative, built into sklearn) ---
# WHY HistGradientBoosting:
#   • Zero extra install — already inside sklearn
#   • Uses histogram-based splitting (same idea as LightGBM / XGBoost)
#     which makes it much faster than standard GradientBoostingClassifier
#   • Handles missing values (NaN) natively — no imputer required for it
#   • l2_regularization acts like XGBoost's reg_lambda — penalises large
#     leaf weights and trims the model to prevent overfitting
#   • class_weight="balanced" handles residual imbalance after ROS
#   • On benchmarks it scores within 1-2% of XGBoost on tabular datasets
models["HistGradientBoosting"] = build_full_pipeline(
    HistGradientBoostingClassifier(
        max_iter=300,              # number of boosting rounds (= n_estimators)
        learning_rate=0.05,        # contribution of each tree — smaller = more precise
        max_depth=6,               # max depth of each tree — controls complexity
        min_samples_leaf=20,       # min samples per leaf — smooths the model
        l2_regularization=1.0,     # L2 penalty on leaf weights (trims the model)
        class_weight="balanced",   # auto-upweights minority classes
        random_state=42,
    )
)
print("  [5] HistGradientBoosting — sklearn's built-in XGBoost alternative")

print(f"\n  Total models to train: {len(models)}")
print()

STEP 10 : Defining candidate models
  [1] LogisticRegression — linear baseline with L2 regularisation
  [2] RandomForest — 200 trees, bagging, feature importance available
  [3] SVM (RBF) — non-linear kernel, good for high-dim audio features
  [4] GradientBoosting — sequential tree boosting, sklearn native
  [5] HistGradientBoosting — sklearn's built-in XGBoost alternative

  Total models to train: 5



In [54]:
# =============================================================================
# STEP 11 — CROSS-VALIDATION ON TRAINING DATA
# =============================================================================
# We evaluate every model using 5-fold Stratified Cross-Validation on the
# TRAINING set only (you've already held out your test set externally).
#
# Why cross-validate instead of just fitting once?
#   A single fit on all training data gives no indication of how the model
#   generalises. CV gives an unbiased estimate by rotating the held-out fold.
#
# Metrics used:
#   accuracy   — overall % of correct predictions
#   f1_macro   — average F1 across all 4 classes equally weighted;
#                better than accuracy for slightly imbalanced data

print("=" * 65)
print("STEP 11 : Cross-validation on training data  (5-fold stratified)")
print("=" * 65)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_summary = {}

for name, pipe in models.items():
    print(f"\n  → Training [{name}] ...")
    scores = cross_validate(
        pipe, X_train_raw, y_train,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        n_jobs=-1,
        return_train_score=False,   # we only care about validation fold scores
    )
    acc_mean = scores["test_accuracy"].mean()
    acc_std  = scores["test_accuracy"].std()
    f1_mean  = scores["test_f1_macro"].mean()
    f1_std   = scores["test_f1_macro"].std()

    cv_summary[name] = {
        "Accuracy Mean": round(acc_mean, 4),
        "Accuracy Std" : round(acc_std,  4),
        "F1 Macro Mean": round(f1_mean,  4),
        "F1 Macro Std" : round(f1_std,   4),
    }
    print(f"     Accuracy  : {acc_mean:.4f}  ± {acc_std:.4f}")
    print(f"     F1 Macro  : {f1_mean:.4f}  ± {f1_std:.4f}")

# Print summary table
summary_df = (
    pd.DataFrame(cv_summary).T
    .sort_values("F1 Macro Mean", ascending=False)
)
print("\n\n  ── Cross-Validation Summary (sorted by F1 Macro) ──")
print(summary_df.to_string())

# Identify best model by F1 macro
best_model_name = summary_df.index[0]
best_f1         = summary_df.loc[best_model_name, "F1 Macro Mean"]
print(f"\n  ✓ Best model : {best_model_name}  (CV F1 Macro = {best_f1:.4f})")
print()

STEP 11 : Cross-validation on training data  (5-fold stratified)

  → Training [LogisticRegression] ...
     Accuracy  : 0.7203  ± 0.0074
     F1 Macro  : 0.7142  ± 0.0077

  → Training [RandomForest] ...
     Accuracy  : 0.9083  ± 0.0065
     F1 Macro  : 0.9108  ± 0.0061

  → Training [SVM_RBF] ...
     Accuracy  : 0.9200  ± 0.0063
     F1 Macro  : 0.9216  ± 0.0068

  → Training [GradientBoosting] ...
     Accuracy  : 0.9103  ± 0.0056
     F1 Macro  : 0.9122  ± 0.0054

  → Training [HistGradientBoosting] ...
     Accuracy  : 0.9213  ± 0.0075
     F1 Macro  : 0.9233  ± 0.0071


  ── Cross-Validation Summary (sorted by F1 Macro) ──
                      Accuracy Mean  Accuracy Std  F1 Macro Mean  F1 Macro Std
HistGradientBoosting         0.9213        0.0075         0.9233        0.0071
SVM_RBF                      0.9200        0.0063         0.9216        0.0068
GradientBoosting             0.9103        0.0056         0.9122        0.0054
RandomForest                 0.9083        0.

In [ ]:
print("=" * 65)
print(f"STEP 12 : Hyperparameter tuning  →  {best_model_name}")
print("=" * 65)

# Param grids for each possible best model
# Keys use double-underscore syntax to navigate into Pipeline steps:
#   "clf__C"            → set C on the "clf" step of the Pipeline
#   "featsel__kbest__k" → set k on the "kbest" step inside the "featsel" step
param_grids = {
    "LogisticRegression": {
        "clf__C"           : [0.1, 1.0, 10.0],
        "featsel__kbest__k": [30, 40, 50],
    },
    "RandomForest": {
        "clf__n_estimators"  : [100, 200],
        "clf__max_depth"     : [10, 20, None],
        "clf__min_samples_leaf": [1, 2],
        "featsel__kbest__k"  : [30, 40, 50],
    },
    "SVM_RBF": {
        "clf__C"           : [1, 10, 50],
        "clf__gamma"       : ["scale", "auto"],
        "featsel__kbest__k": [30, 40],
    },
    "GradientBoosting": {
        "clf__n_estimators" : [100, 200],
        "clf__max_depth"    : [3, 5],
        "clf__learning_rate": [0.05, 0.1],
        "featsel__kbest__k" : [30, 40],
    },
    "HistGradientBoosting": {
        "clf__max_iter"         : [200, 300],          # boosting rounds
        "clf__max_depth"        : [4, 6],              # tree depth
        "clf__learning_rate"    : [0.05, 0.1],         # step size per tree
        "clf__l2_regularization": [0.0, 1.0],          # model trimming strength
        "featsel__kbest__k"     : [35, 40, 45],        # number of features to keep
    },
}

best_pipe  = models[best_model_name]
param_grid = param_grids.get(best_model_name, {})

if param_grid:
    n_combos = 1
    for v in param_grid.values():
        n_combos *= len(v)
    print(f"  Parameter combinations to test : {n_combos}")
    print(f"  × 5 CV folds = {n_combos * 5} total model fits")
    print(f"  Scoring metric : F1 Macro\n")

    gs = GridSearchCV(
        estimator  = best_pipe,
        param_grid = param_grid,
        cv         = cv,
        scoring    = "f1_macro",
        n_jobs     = -1,
        verbose    = 1,
        refit      = True,    # after search, refit best params on ALL training data
    )
    gs.fit(X_train_raw, y_train)

    print(f"\n  Best parameters found:")
    for param, val in gs.best_params_.items():
        print(f"    {param:35s} = {val}")
    print(f"\n  Best CV F1 Macro : {gs.best_score_:.4f}")

    final_model = gs.best_estimator_   # already refit on full training set
else:
    print("  No param grid defined — fitting with default parameters.")
    best_pipe.fit(X_train_raw, y_train)
    final_model = best_pipe

print()


STEP 12 : Hyperparameter tuning  →  HistGradientBoosting
  Parameter combinations to test : 48
  × 5 CV folds = 240 total model fits
  Scoring metric : F1 Macro

Fitting 5 folds for each of 48 candidates, totalling 240 fits


In [ ]:
print("=" * 65)
print("STEP 13 : Training-set fit report  (this is NOT a test evaluation)")
print("=" * 65)
print("  NOTE: Metrics below are on the training data itself.")
print("        Use your held-out test set for unbiased evaluation.\n")
 
y_train_pred = final_model.predict(X_train_raw)
 
print(classification_report(
    y_train, y_train_pred,
    target_names=le.classes_,
    digits=4,
))
 
# Confusion matrix on training set
fig, ax = plt.subplots(figsize=(7, 5))
cm   = confusion_matrix(y_train, y_train_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title(f"Training Set Confusion Matrix — {best_model_name}")


In [ ]:
print("=" * 65)
print("STEP 14 : Feature importances")
print("=" * 65)
 
try:
    clf_step    = final_model.named_steps["clf"]
    prep_step   = final_model.named_steps["prep"]
    featsel_step= final_model.named_steps["featsel"]
 
    if hasattr(clf_step, "feature_importances_"):
        # Reconstruct feature names that survived preprocessing + selection
        all_names_after_prep = num_cols + cat_cols
 
        var_mask   = featsel_step.named_steps["var_filter"].get_support()
        names_after_var = [n for n, m in zip(all_names_after_prep, var_mask) if m]
 
        kbest_mask = featsel_step.named_steps["kbest"].get_support()
        selected_names = [n for n, m in zip(names_after_var, kbest_mask) if m]
 
        importances = clf_step.feature_importances_
 
        feat_df = (
            pd.DataFrame({"Feature": selected_names, "Importance": importances})
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )
 
        print("  Top 15 most important features:")
        print(feat_df.head(15).to_string(index=False))
 
        # Bar plot
        fig, ax = plt.subplots(figsize=(9, 6))
        sns.barplot(
            data=feat_df.head(20),
            y="Feature", x="Importance",
            ax=ax, palette="viridis",
        )
        ax.set_title(f"Top 20 Feature Importances — {best_model_name}")
        ax.set_xlabel("Importance score")
        plt.tight_layout()
        plt.savefig("feature_importance.png", dpi=150)
        plt.close()
        print("\n  Feature importance plot saved → feature_importance.png")
    else:
        print(f"  [{best_model_name}] does not expose feature_importances_.")
        print("  (Logistic Regression / SVM — use coefficients instead if needed)")
 
except Exception as exc:
    print(f"  Could not extract feature importances: {exc}")
 
print()

In [ ]:
print("=" * 65)
print("STEP 15 : Saving the trained model to disk")
print("=" * 65)
 
MODEL_PATH = "best_model.pkl"
LE_PATH    = "label_encoder.pkl"
 
joblib.dump(final_model, MODEL_PATH)
joblib.dump(le,          LE_PATH)
 
print(f"  Trained pipeline  → {MODEL_PATH}")
print(f"  Label encoder     → {LE_PATH}")
print()
print("  To use later on your test set:")
print("    import joblib")
print(f"    model = joblib.load('{MODEL_PATH}')")
print(f"    le    = joblib.load('{LE_PATH}')")
print("    preds = le.inverse_transform(model.predict(X_test))")
print()
print("=" * 65)
print("TRAINING PIPELINE COMPLETE")
print("=" * 65)